# spaGAPA Tutorial: Basic Usage

This notebook walks through the core spaGAPA workflow:
1. Simulate spatial APA data
2. GP imputation with uncertainty
3. APA quantification
4. Spatial domain identification
5. Differential APA analysis
6. SVAPA gene detection
7. Visualization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# spaGAPA imports
from spagapa.benchmark import simulate_spatial_apa
from spagapa.imputation import GPImputer, GPImputerBatch
from spagapa.quantification import APAIndexCalculator
from spagapa.analysis import (
    identify_spatial_domains,
    test_differential_apa,
    find_domain_markers,
    identify_svapa_genes,
)
from spagapa.visualization import (
    plot_spatial_apa,
    plot_spatial_domains,
    plot_volcano,
    plot_heatmap,
)

print('spaGAPA loaded successfully')

## Step 1: Simulate spatial APA data

In [ ]:
# Generate synthetic data with 4 spatial domains
data = simulate_spatial_apa(
    n_spots=300,
    n_genes=60,
    n_domains=4,
    dropout_rate=0.40,
    noise_level=0.08,
    random_state=42
)

coords       = data['coordinates']          # (300, 2)
true_apa     = data['true_apa']             # (300, 60) ground truth
observed_apa = data['observed_apa']         # (300, 60) with NaN
true_domains = data['domain_labels']        # (300,)

# spaGAPA convention: (n_genes, n_spots)
apa_matrix = observed_apa.T
gene_names = [f'Gene_{i:03d}' for i in range(apa_matrix.shape[0])]

dropout_pct = np.isnan(observed_apa).mean() * 100
print(f'Spots: {coords.shape[0]}, Genes: {apa_matrix.shape[0]}')
print(f'Dropout rate: {dropout_pct:.1f}%')

## Step 2: GP Imputation with uncertainty

In [ ]:
# Impute a single gene to demonstrate uncertainty
imputer = GPImputer(kernel_type='matern')
gene0 = apa_matrix[0].copy()
imputed_gene0, uncertainty_gene0 = imputer.impute(gene0, coords)

n_missing = np.isnan(gene0).sum()
print(f'Gene_000: imputed {n_missing} missing values')
print(f'Mean uncertainty: {uncertainty_gene0.mean():.3f}')

# Batch impute all genes
print('\nBatch imputing all genes...')
base_imputer = GPImputer(kernel_type='matern')
batch = GPImputerBatch(
    base_imputer=base_imputer,
    coordinates=coords,
    values=apa_matrix,
    n_jobs=1,
    verbose=False
)
imputed_matrix, uncertainty_matrix = batch.impute()
print(f'Imputed matrix shape: {imputed_matrix.shape}')
print(f'Any NaN remaining: {np.isnan(imputed_matrix).any()}')

## Step 3: APA Quantification

In [ ]:
# Simulate proximal/distal counts from imputed APA index
# In real data these come from scAPAtrap output
total_counts = 100
distal_counts  = (imputed_matrix * total_counts).astype(int)
proximal_counts = total_counts - distal_counts

calc = APAIndexCalculator()

# RUD for gene 0
rud_gene0 = calc.calculate_rud(proximal_counts[0], distal_counts[0])
print(f'Gene_000 RUD: mean={rud_gene0.mean():.3f}, range=[{rud_gene0.min():.3f}, {rud_gene0.max():.3f}]')

# PDUI for gene 0
pdui_gene0 = calc.calculate_pdui(proximal_counts[0], distal_counts[0])
print(f'Gene_000 PDUI: mean={pdui_gene0.mean():.1f}%')

## Step 4: Spatial Domain Identification

In [ ]:
labels, domain_stats = identify_spatial_domains(
    imputed_matrix, coords,
    method='kmeans',
    n_clusters=4,
    refine=True,
    min_domain_size=10
)

print(f'Identified {len(domain_stats)} domains')
print(domain_stats[['domain', 'n_spots', 'mean_apa']].to_string(index=False))

# Compare with true domains
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(true_domains, labels)
print(f'\nAdjusted Rand Index vs true domains: {ari:.3f}')

## Step 5: Differential APA Analysis

In [ ]:
# Compare domain 0 vs domain 1
g1 = np.where(labels == 0)[0]
g2 = np.where(labels == 1)[0]

results = test_differential_apa(
    imputed_matrix, g1, g2,
    gene_names=gene_names,
    method='wilcoxon'
)

sig = results[results['padj'] < 0.05]
print(f'Significant genes (FDR < 0.05): {len(sig)}')
print(sig.nsmallest(5, 'padj')[['gene', 'log2fc', 'padj']].to_string(index=False))

# Domain markers
markers = find_domain_markers(imputed_matrix, labels, gene_names=gene_names)
for d, df in markers.items():
    print(f'Domain {d}: {len(df)} marker genes')

## Step 6: SVAPA Gene Detection

In [ ]:
svapa_genes, svapa_results = identify_svapa_genes(
    imputed_matrix, coords,
    gene_names=gene_names,
    fdr_threshold=0.10
)

print(f'SVAPA genes detected: {len(svapa_genes)}')
if len(svapa_results) > 0:
    top5 = svapa_results.nlargest(5, 'morans_i')
    print(top5[['gene', 'morans_i', 'pvalue']].to_string(index=False))

## Step 7: Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Spatial APA map
ax = axes[0]
sc = ax.scatter(coords[:, 0], coords[:, 1],
                c=imputed_matrix[0], cmap='viridis', s=15, alpha=0.8)
plt.colorbar(sc, ax=ax, label='APA Index')
ax.set_title('Gene_000 APA (imputed)')
ax.set_aspect('equal')

# 2. Domain map
ax = axes[1]
sc = ax.scatter(coords[:, 0], coords[:, 1],
                c=labels, cmap='tab10', s=15, alpha=0.8)
plt.colorbar(sc, ax=ax, label='Domain')
ax.set_title('Spatial Domains')
ax.set_aspect('equal')

# 3. Volcano plot
ax = axes[2]
neg_log_p = -np.log10(results['padj'].values + 1e-300)
colors = np.where(results['padj'] < 0.05, 'red', 'gray')
ax.scatter(results['log2fc'], neg_log_p, c=colors, s=15, alpha=0.6)
ax.axhline(-np.log10(0.05), color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('log2 Fold Change')
ax.set_ylabel('-log10(padj)')
ax.set_title('Differential APA (Domain 0 vs 1)')

plt.tight_layout()
plt.savefig('tutorial_output.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: tutorial_output.png')

## Summary

In this tutorial we:
- Simulated spatial APA data with 300 spots, 60 genes, 40% dropout
- Imputed missing values using Gaussian process (Matérn kernel)
- Calculated RUD and PDUI APA indices
- Identified 4 spatial domains with K-means
- Found differential APA genes between domains
- Detected spatially variable APA genes (SVAPA)

For real data, replace the simulation step with loading your BAM file and coordinates.